# Student Dropout Prediction
## Student 3 – Model Optimization

### Objectives
1. Machine Learning Pipeline
2. 5-Fold Cross Validation
3. GridSearchCV
4. Hyperparameter Tuning
5. Best Model Selection
6. Save Best Model using Pickle

In [12]:
import pandas as pd
import numpy as np
import pickle
import warnings

warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_val_score
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("All libraries imported successfully!")

All libraries imported successfully!


In [13]:
df = pd.read_csv("../data.csv", sep=';')

print("Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)

df.head()

Dataset Loaded Successfully!
Dataset Shape: (4424, 37)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [14]:
print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDataset Shape:")
print(df.shape)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance	                     4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualificati

In [15]:
print("Dataset Columns:")
print(df.columns.tolist())

Dataset Columns:
['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance\t', 'Previous qualification', 'Previous qualification (grade)', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Admission grade', 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Unemployment rate', 'Inflation rate', 'GDP', 'Tar

In [16]:
# Separate Features and Target

X = df.drop('Target', axis=1)
y = df['Target']

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nTarget Classes:")
print(y.value_counts())

Features Shape: (4424, 36)
Target Shape: (4424,)

Target Classes:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


In [17]:
print("Missing Values in Features:", X.isnull().sum().sum())
print("Missing Values in Target:", y.isnull().sum())

Missing Values in Features: 0
Missing Values in Target: 0


In [18]:
print("Data Types:")
print(X.dtypes.value_counts())

print("\nCategorical Columns:")
print(X.select_dtypes(include=['object']).columns.tolist())

Data Types:
int64      29
float64     7
Name: count, dtype: int64

Categorical Columns:
[]


In [19]:
print("Target Distribution:")
print(y.value_counts())

print("\nTarget Percentage:")
print(y.value_counts(normalize=True) * 100)

Target Distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Target Percentage:
Target
Graduate    49.932188
Dropout     32.120253
Enrolled    17.947559
Name: proportion, dtype: float64


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (3539, 36)
Testing Data: (885, 36)


In [21]:
print("Training Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

Training Target Distribution:
Target
Graduate    1767
Dropout     1137
Enrolled     635
Name: count, dtype: int64

Testing Target Distribution:
Target
Graduate    442
Dropout     284
Enrolled    159
Name: count, dtype: int64


In [22]:
# Logistic Regression Pipeline
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000))
])

# Decision Tree Pipeline
dt_pipeline = Pipeline([
    ('model', DecisionTreeClassifier(random_state=42))
])

# Random Forest Pipeline
rf_pipeline = Pipeline([
    ('model', RandomForestClassifier(random_state=42))
])

# KNN Pipeline
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

print("All Machine Learning Pipelines Created Successfully!")

All Machine Learning Pipelines Created Successfully!


In [23]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': lr_pipeline,
    'Decision Tree': dt_pipeline,
    'Random Forest': rf_pipeline,
    'KNN': knn_pipeline
}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='f1_macro'
    )

    print("\n", name)
    print("Fold Scores:", scores)
    print("Mean F1 Score:", scores.mean())


 Logistic Regression
Fold Scores: [0.69776641 0.68343087 0.67919108 0.68113861 0.63980035]
Mean F1 Score: 0.6762654650365153

 Decision Tree
Fold Scores: [0.63813982 0.62665827 0.63404477 0.61720626 0.61712417]
Mean F1 Score: 0.6266346560084569

 Random Forest
Fold Scores: [0.71503486 0.66498766 0.69219943 0.65060457 0.70303538]
Mean F1 Score: 0.6851723808500556

 KNN
Fold Scores: [0.62396168 0.60704492 0.65012779 0.58470091 0.58507214]
Mean F1 Score: 0.6101814888741834


In [24]:
lr_params = {
    'model__C': [0.1, 1, 10],
    'model__solver': ['lbfgs', 'liblinear']
}

lr_grid = GridSearchCV(
    lr_pipeline,
    lr_params,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1
)

lr_grid.fit(X_train, y_train)

print("Best Parameters:", lr_grid.best_params_)
print("Best F1 Score:", lr_grid.best_score_)

Best Parameters: {'model__C': 1, 'model__solver': 'lbfgs'}
Best F1 Score: 0.6762654650365153


In [25]:
dt_params = {
    'model__max_depth': [5, 10, 20, None],
    'model__min_samples_split': [2, 5, 10],
    'model__criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(
    dt_pipeline,
    dt_params,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

print("Best Parameters:", dt_grid.best_params_)
print("Best F1 Score:", dt_grid.best_score_)

Best Parameters: {'model__criterion': 'gini', 'model__max_depth': 10, 'model__min_samples_split': 5}
Best F1 Score: 0.6576787205070976


In [26]:
rf_params = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20, None],
    'model__min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_params,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best Parameters:", rf_grid.best_params_)
print("Best F1 Score:", rf_grid.best_score_)

Best Parameters: {'model__max_depth': 20, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best F1 Score: 0.6907430727347211


In [27]:
knn_params = {
    'model__n_neighbors': [3, 5, 7, 9],
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2]
}

knn_grid = GridSearchCV(
    knn_pipeline,
    knn_params,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1
)

knn_grid.fit(X_train, y_train)

print("Best Parameters:", knn_grid.best_params_)
print("Best F1 Score:", knn_grid.best_score_)

Best Parameters: {'model__n_neighbors': 5, 'model__p': 1, 'model__weights': 'uniform'}
Best F1 Score: 0.6330529714590358


In [28]:
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Decision Tree',
        'Random Forest',
        'KNN'
    ],
    'Best CV F1 Score': [
        lr_grid.best_score_,
        dt_grid.best_score_,
        rf_grid.best_score_,
        knn_grid.best_score_
    ]
})

results = results.sort_values(
    by='Best CV F1 Score',
    ascending=False
)

print(results)

                 Model  Best CV F1 Score
2        Random Forest          0.690743
0  Logistic Regression          0.676265
1        Decision Tree          0.657679
3                  KNN          0.633053


In [29]:
best_grids = {
    'Logistic Regression': lr_grid,
    'Decision Tree': dt_grid,
    'Random Forest': rf_grid,
    'KNN': knn_grid
}

best_model_name = max(
    best_grids,
    key=lambda name: best_grids[name].best_score_
)

best_model = best_grids[best_model_name].best_estimator_

print("Best Model:", best_model_name)
print("Best CV F1 Score:", best_grids[best_model_name].best_score_)

Best Model: Random Forest
Best CV F1 Score: 0.6907430727347211


In [30]:
y_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Test Accuracy: 0.7706214689265537

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.82      0.76      0.79       284
    Enrolled       0.56      0.36      0.44       159
    Graduate       0.79      0.93      0.85       442

    accuracy                           0.77       885
   macro avg       0.72      0.68      0.69       885
weighted avg       0.76      0.77      0.76       885



In [31]:
results.to_csv("../optimization_results.csv", index=False)

print("Optimization Results Saved Successfully!")

Optimization Results Saved Successfully!


In [32]:
with open("../best_model.pkl", "wb") as file:
    pickle.dump(best_model, file)

print("Best Model Saved Successfully!")

Best Model Saved Successfully!
